# Fragmentação de leis com o Qwen 2.5 — GPU A100

Passa as leis de `coleta/` pelo Qwen 2.5 7B oficial, em precisão cheia, e grava a resposta de cada uma em JSON no Google Drive.

1. **Ambiente de execução > Alterar o tipo de ambiente de execução > GPU A100.** Exige unidades de computação. O saldo e o consumo por hora aparecem no painel de recursos da sessão.
2. O modelo é de acesso aberto: não exige licença nem token.
3. Rode as células na ordem. Se, depois da instalação, o Colab pedir para reiniciar a sessão, reinicie e rode tudo de novo desde a primeira célula.
4. `LIMITE = 300` processa uma amostra fixa de 300 leis, a mesma em toda execução e nos notebooks dos três modelos. Anote a velocidade (tokens/s) e o consumo de unidades. Para o acervo inteiro, ponha `LIMITE = None` na primeira célula, rode-a e rode de novo a célula **Execução**: o que já foi respondido é pulado.

Se a sessão cair, basta rodar tudo de novo — o trabalho continua de onde parou. **Ao terminar, desconecte o ambiente de execução**, para não gastar unidades com a GPU parada.

São duas medidas, em pastas separadas do Drive, escolhidas por `RESTRITA` na primeira célula:

- `RESTRITA = True` — o motor obriga a resposta a ser um JSON no formato pedido. É a rodada que se compara com o regex.
- `RESTRITA = False` — o modelo escreve livremente. Serve para medir, numa amostra, quantas respostas saem como JSON válido.

A opção `SEM_ESPACOS` fica desligada (`False`). Ligada, ela tira da gramática da rodada restrita o espaço em branco livre entre os elementos do JSON, onde o modelo às vezes entra num laço até esgotar a resposta, e manda a rodada para uma pasta própria (`..._restrita_sem_espacos`). No teste com a amostra do Llama o laço de espaços sumiu, mas o modelo passou a repetir outros caracteres dentro dos textos e a concordância com o regex caiu.


In [ ]:
MODELO = "Qwen/Qwen2.5-7B-Instruct"   # o modelo oficial, em precisão cheia
NOME = "qwen_oficial"   # prefixo das pastas de saída no Drive

# True: o motor só deixa o modelo escrever JSON válido no formato pedido (mede o conteúdo).
# False: o modelo escreve livremente (mede quantas respostas saem como JSON válido).
RESTRITA = True
# True: na rodada restrita, a gramática não aceita espaço em branco livre entre os elementos
# do JSON, onde o modelo às vezes entra em laço. A rodada vai para uma pasta própria.
SEM_ESPACOS = False
SAIDA = (f"/content/drive/MyDrive/fragmentos_leis/{NOME}_" + ("restrita" if RESTRITA else "livre")
         + ("_sem_espacos" if RESTRITA and SEM_ESPACOS else ""))

LIMITE = 300            # as primeiras N leis de uma amostra fixa; None = todas as que faltam
MAX_TOKENS_LEI = 9000   # a lei maior que isto fica para depois (lista em longas.txt); None = sem limite
TRECHO = 3000           # a lei maior que isto é dividida em trechos deste tamanho, em tokens
JANELA = 12288          # prompt + resposta, em tokens
BLOCO = 256             # pedidos (lei inteira ou trecho) gerados em paralelo; a cada bloco os resultados vão para o Drive
DTYPE = "auto"      # tipo numérico dos cálculos; "auto" usa o do modelo
MEMORIA_GPU = 0.90      # fração da memória da GPU entregue ao motor

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content
!test -d extrator_fragmentos_leis || git clone --depth 1 https://github.com/filipemcbarros/extrator_fragmentos_leis.git
!pip install -q vllm lxml

## Prompt

O mesmo texto para todas as leis: a tarefa, o formato da resposta e dois exemplos. A lei entra no fim. A célula grava o texto num arquivo, que o programa abaixo lê.

In [ ]:
%%writefile /content/prompt.txt
Você recebe o texto de uma lei brasileira, em HTML, e deve devolvê-lo fragmentado em seus dispositivos, em JSON.

A hierarquia segue a Lei Complementar 95/1998:
- agrupadores, do maior para o menor: parte > livro > titulo > capitulo > secao > subsecao;
- artigo ("Art. 1º", "Art. 10."), dentro do agrupador aberto, ou na raiz se não houver;
- paragrafo ("§ 1º", "Parágrafo único") e inciso ("I -", "II -") pendem do artigo;
- o inciso que vem depois de um parágrafo pende desse parágrafo;
- alinea ("a)", "b)") pende do inciso; item ("1.", "2.") pende da alínea.

Responda com um único objeto JSON, sem comentários e sem cercas de código, com as chaves:
- "titulo": o nome da lei ("LEI Nº 1.234, DE 5 DE MAIO DE 1990");
- "ementa": o resumo que vem logo abaixo do título;
- "preambulo": o que vem entre a ementa e o primeiro dispositivo ("O PRESIDENTE DA REPÚBLICA Faço saber...");
- "fecho": local, data e assinaturas, depois do último dispositivo;
- "dispositivos": a lista dos dispositivos de primeiro nível.

Cada dispositivo é um objeto com:
- "tipo": parte, livro, titulo, capitulo, secao, subsecao, artigo, paragrafo, inciso, alinea ou item;
- "numero": a numeração como está na lei, sem o rótulo: "1º", "10", "5º-A", "II", "a", "único";
- "rubrica": só nos agrupadores, o nome que os acompanha ("DAS DISPOSIÇÕES GERAIS");
- "texto": o texto do dispositivo sem o rótulo; no artigo, é o caput; no agrupador, fica vazio;
- "filhos": os dispositivos subordinados, na ordem; omita a chave quando não houver.

Regras:
- copie o texto exatamente como está, sem corrigir grafia, resumir ou traduzir;
- todo o texto da lei deve aparecer na resposta, cada trecho uma única vez;
- crie só os dispositivos que estão escritos na lei: nenhum dispositivo vazio, e nenhum agrupador (parte, livro, titulo, capitulo, secao, subsecao) que a lei não traga. Na lei sem agrupadores, os artigos vão direto em "dispositivos";
- o artigo sem parágrafos nem incisos não tem "filhos";
- quando a lei altera outra e transcreve entre aspas o texto alterado, a transcrição inteira é texto do artigo que faz a alteração: não crie dispositivos para o que está entre aspas.

Exemplo 1 de entrada:
<h1>LEI Nº 8.888, DE 7 DE JUNHO DE 1994</h1>
<div class="textoNorma"><p class="ementa">Institui o Dia Nacional do Livro.</p>
<div class="texto"><p>O PRESIDENTE DA REPÚBLICA<br>Faço saber que o Congresso Nacional decreta e eu sanciono a seguinte Lei:</p>
<p>Art. 1º É instituído o Dia Nacional do Livro, a ser comemorado anualmente em 29 de outubro.</p>
<p>Art. 2º Esta Lei entra em vigor na data de sua publicação.</p>
<p>Brasília, 7 de junho de 1994; 173º da Independência e 106º da República.<br>FULANO DE TAL<br>Beltrano de Tal</p></div></div>

Resposta do exemplo 1:
{"titulo":"LEI Nº 8.888, DE 7 DE JUNHO DE 1994","ementa":"Institui o Dia Nacional do Livro.","preambulo":"O PRESIDENTE DA REPÚBLICA Faço saber que o Congresso Nacional decreta e eu sanciono a seguinte Lei:","fecho":"Brasília, 7 de junho de 1994; 173º da Independência e 106º da República. FULANO DE TAL Beltrano de Tal","dispositivos":[{"tipo":"artigo","numero":"1º","texto":"É instituído o Dia Nacional do Livro, a ser comemorado anualmente em 29 de outubro."},{"tipo":"artigo","numero":"2º","texto":"Esta Lei entra em vigor na data de sua publicação."}]}

Exemplo 2 de entrada:
<h1>LEI Nº 9.999, DE 3 DE MARÇO DE 1999</h1>
<div class="textoNorma"><p class="ementa">Dispõe sobre o registro de embarcações e altera a Lei nº 1.111, de 1950.</p>
<div class="texto"><p>O PRESIDENTE DA REPÚBLICA<br>Faço saber que o Congresso Nacional decreta e eu sanciono a seguinte Lei:</p>
<p>CAPÍTULO I<br>DO REGISTRO</p>
<p>Art. 1º O registro de embarcações é obrigatório:<br>I - para as de pesca;<br>II - para as de transporte:<br>a) de carga;<br>b) de passageiros.<br>Parágrafo único. O registro é gratuito.</p>
<p>Art. 2º O art. 4º da Lei nº 1.111, de 1950, passa a vigorar com a seguinte redação:<br>"Art. 4º O prazo é de trinta dias.<br>§ 1º O prazo pode ser prorrogado." (NR)</p>
<p>Art. 3º Esta Lei entra em vigor na data de sua publicação.</p>
<p>Brasília, 3 de março de 1999; 178º da Independência e 111º da República.<br>FULANO DE TAL<br>Beltrano de Tal</p></div></div>

Resposta do exemplo 2:
{"titulo":"LEI Nº 9.999, DE 3 DE MARÇO DE 1999","ementa":"Dispõe sobre o registro de embarcações e altera a Lei nº 1.111, de 1950.","preambulo":"O PRESIDENTE DA REPÚBLICA Faço saber que o Congresso Nacional decreta e eu sanciono a seguinte Lei:","fecho":"Brasília, 3 de março de 1999; 178º da Independência e 111º da República. FULANO DE TAL Beltrano de Tal","dispositivos":[{"tipo":"capitulo","numero":"I","rubrica":"DO REGISTRO","texto":"","filhos":[{"tipo":"artigo","numero":"1º","texto":"O registro de embarcações é obrigatório:","filhos":[{"tipo":"inciso","numero":"I","texto":"para as de pesca;"},{"tipo":"inciso","numero":"II","texto":"para as de transporte:","filhos":[{"tipo":"alinea","numero":"a","texto":"de carga;"},{"tipo":"alinea","numero":"b","texto":"de passageiros."}]},{"tipo":"paragrafo","numero":"único","texto":"O registro é gratuito."}]},{"tipo":"artigo","numero":"2º","texto":"O art. 4º da Lei nº 1.111, de 1950, passa a vigorar com a seguinte redação: \"Art. 4º O prazo é de trinta dias. § 1º O prazo pode ser prorrogado.\" (NR)"},{"tipo":"artigo","numero":"3º","texto":"Esta Lei entra em vigor na data de sua publicação."}]}]}

Agora a lei a fragmentar:

## Programa

A célula grava o programa em `/content/fragmentar.py`; quem o roda é a célula seguinte. O vLLM não funciona dentro do notebook, só como programa à parte.

Para cada lei: recorta da página o título e o bloco `div.textoNorma` (o resto são 48 KB de menu e rodapé), monta o prompt, gera a resposta e grava.

A lei com mais de `TRECHO` tokens não cabe na janela junto com a resposta e vai dividida. O corte é feito nas quebras de linha e nos fins de bloco do HTML, nunca nos rótulos, para não entregar ao modelo parte do trabalho que se quer medir. Cada trecho é um pedido à parte, com um aviso no prompt dizendo que trecho é; as árvores dos trechos são emendadas depois, na comparação (`modelos/resposta.py`). A lei com mais de `MAX_TOKENS_LEI` tokens fica de fora.

In [ ]:
%%writefile /content/fragmentar.py
import argparse, json, pathlib, random, re, time
from lxml import etree, html as lhtml

COLETA = pathlib.Path("/content/extrator_fragmentos_leis/coleta")

# O formato da resposta, para a decodificação restrita. É o mesmo que o prompt
# descreve: o dispositivo contém dispositivos, daí a referência a si mesmo.
ESQUEMA = {
    "type": "object",
    "properties": {
        "titulo": {"type": "string"},
        "ementa": {"type": "string"},
        "preambulo": {"type": "string"},
        "fecho": {"type": "string"},
        "dispositivos": {"type": "array", "items": {"$ref": "#/$defs/dispositivo"}},
    },
    "required": ["titulo", "ementa", "preambulo", "fecho", "dispositivos"],
    "additionalProperties": False,
    "$defs": {
        "dispositivo": {
            "type": "object",
            "properties": {
                "tipo": {"enum": ["parte", "livro", "titulo", "capitulo", "secao", "subsecao",
                                  "artigo", "paragrafo", "inciso", "alinea", "item"]},
                "numero": {"type": "string"},
                "rubrica": {"type": "string"},
                "texto": {"type": "string"},
                "filhos": {"type": "array", "items": {"$ref": "#/$defs/dispositivo"}},
            },
            "required": ["tipo", "numero", "texto"],
            "additionalProperties": False,
        },
    },
}


# O trecho que começa no meio da lei pode abrir com o resto do dispositivo anterior.
ESQUEMA_TRECHO = {**ESQUEMA, "properties": {"continuacao": {"type": "string"}, **ESQUEMA["properties"]}}

# Onde a lei pode ser cortada: depois de uma quebra de linha ou do fim de um bloco.
# São marcas da diagramação, e não dos rótulos — dividir pelos artigos seria
# entregar ao modelo parte do trabalho que se quer medir.
QUEBRA = re.compile(r"(?<=<br>)|(?<=</p>)|(?<=</table>)|(?<=\n)")

# O que o prompt ganha quando a lei vai dividida. O texto é o de modelos/prompt.py.
FIM_DO_PROMPT = "Agora a lei a fragmentar:\n"
PRIMEIRO_TRECHO = """\
Atenção: esta lei é longa e foi dividida em {trechos} trechos. Este é o primeiro. Fragmente só o que está nele; o fecho vem em outro trecho, deixe "fecho" vazio.
"""
TRECHO_SEGUINTE = """\
Atenção: esta lei é longa e foi dividida em {trechos} trechos. Este é o trecho {trecho}, que começa no meio da lei.
- Deixe "titulo", "ementa" e "preambulo" vazios.{sem_fecho}
- Se o trecho começar por texto que continua o dispositivo anterior, antes de qualquer rótulo, ponha esse texto na chave "continuacao".
- O dispositivo cujo pai ficou em trecho anterior (um inciso sem o seu artigo, por exemplo) vai direto em "dispositivos".
"""

# Linha que separa, no arquivo bruto da lei dividida, a resposta de cada trecho.
MARCA = "===== trecho {} de {} =====\n"
# Tokens que o motor acrescenta à mensagem (marcas de início, de papel e de turno).
FOLGA = 128


def recortar(arquivo):
    """Título e bloco da lei, sem scripts, comentários e atributos (menos `class`)."""
    pagina = lhtml.fromstring(arquivo.read_text(encoding="utf-8"))
    titulos = (" ".join(h.text_content().split()) for h in pagina.xpath("//h1"))
    titulo = next((t for t in titulos if any(c.isdigit() for c in t)), "")

    bloco = pagina.xpath('//div[contains(@class,"textoNorma")]')[0]
    etree.strip_elements(bloco, etree.Comment, "script", "style", with_tail=False)
    for e in bloco.iter():
        classe = e.get("class")
        e.attrib.clear()
        if classe:
            e.set("class", classe)
    texto = lhtml.tostring(bloco, encoding="unicode").replace("\xa0", " ")
    texto = re.sub(r"\s*\n\s*", "\n", re.sub(r"[ \t\r]+", " ", texto)).strip()
    return f"<h1>{titulo}</h1>\n{texto}"


def dividir(lei, contar, limite):
    """Divide a lei em trechos de até `limite` tokens. A lei curta sai inteira."""
    if contar(lei) <= limite:
        return [lei]
    trechos, atual, tokens = [], [], 0
    for segmento in QUEBRA.split(lei):
        for pedaco in caber(segmento, contar, limite):
            n = contar(pedaco)
            if atual and tokens + n > limite:
                trechos.append("".join(atual))
                atual, tokens = [], 0
            atual.append(pedaco)
            tokens += n
    if atual:
        trechos.append("".join(atual))
    return trechos


def caber(segmento, contar, limite):
    """Parte por palavras o segmento que sozinho excede o limite (linha sem quebra)."""
    n = contar(segmento)
    if n <= limite:
        return [segmento] if segmento else []
    palavras = segmento.split(" ")
    passo = max(1, len(palavras) * limite // (n + limite))
    return [" ".join(palavras[i:i + passo]) + " " for i in range(0, len(palavras), passo)]


def montar(prompt, trecho, k, n):
    """O prompt da lei inteira (n = 1) ou do trecho k de n."""
    if n == 1:
        return prompt + trecho
    if k == 1:
        aviso = PRIMEIRO_TRECHO.format(trechos=n)
    else:
        sem_fecho = "" if k == n else '\n- O fecho vem em outro trecho: deixe "fecho" vazio.'
        aviso = TRECHO_SEGUINTE.format(trecho=k, trechos=n, sem_fecho=sem_fecho)
    return prompt[:-len(FIM_DO_PROMPT)] + aviso + "\nAgora o trecho a fragmentar:\n" + trecho


def ler_json(resposta):
    """O objeto JSON da resposta, ou None se ela não trouxer um JSON válido."""
    try:
        return json.JSONDecoder().raw_decode(resposta[resposta.index("{"):])[0]
    except ValueError:
        return None


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--modelo", required=True)
    parser.add_argument("--saida", required=True)
    parser.add_argument("--limite", type=int, default=0, help="0 = todas as que faltam")
    parser.add_argument("--max-tokens-lei", type=int, default=9000, help="0 = sem limite")
    parser.add_argument("--trecho", type=int, default=3000, help="tokens de cada trecho da lei dividida")
    parser.add_argument("--janela", type=int, default=12288)
    parser.add_argument("--bloco", type=int, default=256, help="pedidos gerados em paralelo")
    parser.add_argument("--restrita", type=int, default=1, help="1 = decodificação restrita ao esquema")
    parser.add_argument("--sem-espacos", type=int, default=0,
                        help="1 = a gramática não aceita espaço em branco livre no JSON")
    parser.add_argument("--dtype", default="auto")
    parser.add_argument("--memoria", type=float, default=0.90)
    args = parser.parse_args()

    prompt = pathlib.Path("/content/prompt.txt").read_text(encoding="utf-8").rstrip("\n") + "\n"
    assert prompt.endswith(FIM_DO_PROMPT), "O prompt deve terminar em: " + FIM_DO_PROMPT
    saida = pathlib.Path(args.saida)
    saida.mkdir(parents=True, exist_ok=True)
    lista_longas = saida / "longas.txt"
    registro = saida / "registro.csv"

    def gravar(arquivo, pasta, extensao, texto):
        caminho = saida / pasta / arquivo.parent.parent.name / f"{arquivo.stem}.{extensao}"
        caminho.parent.mkdir(parents=True, exist_ok=True)
        caminho.write_text(texto, encoding="utf-8")

    # Feita é a lei que já tem resposta no Drive.
    feitas = {p.stem for p in (saida / "bruto").glob("*/*.txt")}
    leis = sorted(COLETA.glob("*/html/*.html"))
    if args.limite:
        # A amostra sai do acervo inteiro, e não do que falta fazer: é a mesma
        # em toda execução e para todos os modelos.
        leis = random.Random(0).sample(leis, min(args.limite, len(leis)))
    pendentes = [a for a in leis if a.stem not in feitas]
    print(f"{len(leis)} leis nesta rodada, {len(pendentes)} sem resposta.", flush=True)
    if not pendentes:
        return
    # Das menores para as maiores: o bloco só termina quando a resposta mais longa
    # termina, e com tamanhos parecidos a GPU não fica esperando por uma só.
    pendentes.sort(key=lambda a: a.stat().st_size)

    import vllm
    from vllm import LLM, SamplingParams

    restringir, motor = None, {}
    if args.restrita:
        # O nome do recurso mudou entre versões do vLLM.
        try:
            from vllm.sampling_params import StructuredOutputsParams
            restringir = lambda esquema: {"structured_outputs": StructuredOutputsParams(json=esquema)}
        except ImportError:
            if args.sem_espacos:
                raise SystemExit("--sem-espacos exige um vLLM com structured_outputs.")
            from vllm.sampling_params import GuidedDecodingParams
            restringir = lambda esquema: {"guided_decoding": GuidedDecodingParams(json=esquema)}
        if args.sem_espacos:
            # Quem monta a gramática é o motor, e não cada pedido: a opção entra na criação
            # dele. Só os motores de gramática xgrammar e guidance a aceitam.
            motor = {"structured_outputs_config": {"backend": "xgrammar",
                                                   "disable_any_whitespace": True}}
    restricao = restringir(ESQUEMA) if restringir else {}
    restricao_do_trecho = restringir(ESQUEMA_TRECHO) if restringir else {}

    llm = LLM(model=args.modelo, max_model_len=args.janela, dtype=args.dtype,
              gpu_memory_utilization=args.memoria, seed=0, **motor)
    tokenizador = llm.get_tokenizer()

    modo = "livre"
    if restricao:
        modo = f"restrita ({next(iter(restricao))}" + (", sem espaço livre" if motor else "") + ")"
    print(f"vLLM {vllm.__version__}. Decodificação {modo}", flush=True)

    def contar(texto):
        return len(tokenizador.encode(texto))

    def gerar(lote):
        """Gera e grava as respostas de um lote de (arquivo, trechos). Devolve (tokens, segundos)."""
        conversas, parametros = [], []
        for arquivo, trechos in lote:
            for k, trecho in enumerate(trechos, 1):
                conteudo = montar(prompt, trecho, k, len(trechos))
                conversas.append([{"role": "user", "content": conteudo}])
                # A resposta repete o texto da lei e acrescenta as chaves do JSON; o teto é o
                # que sobra da janela. temperature=0: a mesma lei produz sempre a mesma resposta.
                teto = args.janela - contar(conteudo) - FOLGA
                parametros.append(SamplingParams(
                    temperature=0, max_tokens=max(16, min(int(2.2 * contar(trecho)) + 400, teto)),
                    **(restricao if k == 1 else restricao_do_trecho)))

        inicio = time.time()
        respostas = iter(llm.chat(conversas, parametros, use_tqdm=False))
        segundos = time.time() - inicio

        tokens = 0
        for arquivo, trechos in lote:
            saidas = [next(respostas).outputs[0] for _ in trechos]
            objetos = [ler_json(s.text) for s in saidas]
            validos = sum(o is not None for o in objetos)
            gerados = sum(len(s.token_ids) for s in saidas)
            tokens += gerados
            # A lei dividida só vale como JSON se todos os trechos valem; o arquivo
            # traz então a lista deles, na ordem, para a emenda feita na comparação.
            if validos == len(saidas):
                gravar(arquivo, "json", "json",
                       json.dumps(objetos[0] if len(objetos) == 1 else objetos, ensure_ascii=False, indent=2))
            with registro.open("a") as f:
                f.write(f"{arquivo.stem},{gerados},{int(any(s.finish_reason == 'length' for s in saidas))},"
                        f"{int(validos == len(saidas))},{len(saidas)},{validos}\n")
            bruto = saidas[0].text
            if len(saidas) > 1:
                bruto = "".join(MARCA.format(k, len(saidas)) + s.text + "\n" for k, s in enumerate(saidas, 1))
            # Por último: é a existência deste arquivo que marca a lei como feita.
            gravar(arquivo, "bruto", "txt", bruto)
        return tokens, segundos

    # A lista das longas é refeita a cada execução: o limite pode ter mudado.
    lista_longas.write_text("")
    lote, pedidos = [], 0
    for vistas, arquivo in enumerate(pendentes, 1):
        lei = recortar(arquivo)
        if args.max_tokens_lei and contar(lei) > args.max_tokens_lei:
            with lista_longas.open("a") as f:
                f.write(arquivo.stem + "\n")
        else:
            trechos = dividir(lei, contar, args.trecho)
            lote.append((arquivo, trechos))
            pedidos += len(trechos)
        if lote and (pedidos >= args.bloco or vistas == len(pendentes)):
            tokens, segundos = gerar(lote)
            print(f"{vistas}/{len(pendentes)} leis — {pedidos} pedidos, "
                  f"{segundos:.0f} s, {tokens / max(segundos, 1):.0f} tokens/s", flush=True)
            lote, pedidos = [], 0


# O vLLM abre processos auxiliares que reimportam este arquivo: sem esta guarda,
# cada um deles recomeçaria a execução.
if __name__ == "__main__":
    main()

## Execução

Na primeira vez baixa cerca de 15 GB e leva alguns minutos para carregar o modelo; o vLLM imprime muitas linhas de registro nesse meio-tempo. Para cada lei ficam, na pasta do Drive:

- `bruto/<ano>/<slug>.txt` — a resposta como veio; na lei dividida, a de cada trecho, depois de uma linha `===== trecho 2 de 5 =====`;
- `json/<ano>/<slug>.json` — a mesma resposta, quando é um JSON válido; na lei dividida, a lista das respostas dos trechos, quando todas são;
- `registro.csv` — lei, tokens gerados, se alguma resposta foi cortada pelo limite (1), se o JSON é válido (1), número de trechos e quantos deles são JSON válido;
- `longas.txt` — as leis que a última execução deixou de fora por passarem de `MAX_TOKENS_LEI`.

Ao fim de cada bloco sai a velocidade em **tokens/s**. O acervo inteiro pede cerca de 34 milhões de tokens de resposta (uns 10 milhões sem as leis longas): a divisão dá o tempo total.

In [ ]:
!python -u /content/fragmentar.py --modelo "{MODELO}" --saida "{SAIDA}" --limite {LIMITE or 0} --max-tokens-lei {MAX_TOKENS_LEI or 0} --trecho {TRECHO} --janela {JANELA} --restrita {int(RESTRITA)} --sem-espacos {int(SEM_ESPACOS)} --bloco {BLOCO} --dtype {DTYPE} --memoria {MEMORIA_GPU}

## Conferência

Quantas respostas são JSON válido, a contagem de artigos ao lado da do regex numa amostra de 300 leis e uma lei de exemplo.

In [ ]:
import csv, json, pathlib, random

saida = pathlib.Path(SAIDA)
coleta = pathlib.Path("/content/extrator_fragmentos_leis/coleta")


def contar_artigos(dispositivos):
    return sum((d.get("tipo") == "artigo") + contar_artigos(d.get("filhos") or [])
               for d in dispositivos if isinstance(d, dict))


# As contagens saem do registro, um arquivo só; vale a última linha de cada lei.
with (saida / "registro.csv").open(encoding="utf-8") as f:
    registro = {linha[0]: linha for linha in csv.reader(f) if linha}
validas = sorted(slug for slug, linha in registro.items() if linha[3] == "1")
cortadas = sum(linha[2] == "1" for linha in registro.values())
divididas = sum(len(linha) > 4 and int(linha[4]) > 1 for linha in registro.values())
print(f"{len(registro)} leis respondidas, {len(validas)} com JSON válido, {cortadas} cortadas pelo limite, "
      f"{divididas} divididas em trechos.\n")

# A contagem de artigos sai de uma amostra: abrir no Drive todas as respostas leva horas.
ano = {a.stem: a.parent.parent.name for a in coleta.glob("*/html/*.html")}
amostra = random.Random(0).sample(validas, min(300, len(validas)))
iguais = 0
for slug in amostra:
    do_regex = json.loads((coleta / ano[slug] / "json" / f"{slug}.json").read_text(encoding="utf-8"))
    do_modelo = json.loads((saida / "json" / ano[slug] / f"{slug}.json").read_text(encoding="utf-8"))
    # A lei dividida vem como a lista das respostas dos trechos.
    trechos = do_modelo if isinstance(do_modelo, list) else [do_modelo]
    artigos = (do_regex["contagem"].get("artigo", 0),
               sum(contar_artigos(t.get("dispositivos") or []) for t in trechos))
    iguais += artigos[0] == artigos[1]
    if len(amostra) <= 60:
        print(f"{slug:45s} artigos: regex {artigos[0]:3d}  {NOME} {artigos[1]:3d}")
print(f"\nMesmo número de artigos que o regex em {iguais} de {len(amostra)} leis da amostra.\n")

if amostra:
    exemplo = random.choice(amostra)
    print(exemplo)
    print((saida / "json" / ano[exemplo] / f"{exemplo}.json").read_text(encoding="utf-8")[:3000])

## Pacote

Junta num único zip, na pasta `fragmentos_leis` do Drive, as respostas gravadas depois que o programa passou a dividir as leis longas, mais o `registro.csv` e o `longas.txt`. Baixar esse arquivo é mais rápido que baixar a pasta inteira, e não esbarra na cota de operações do Drive. O zip traz a mesma estrutura da pasta: é só extrair em `modelos/resultados/acervo/`, juntando à pasta da rodada que já está lá.

In [ ]:
import csv, pathlib, zipfile

saida = pathlib.Path(SAIDA)
coleta = pathlib.Path("/content/extrator_fragmentos_leis/coleta")

# As linhas novas do registro têm seis colunas; as das rodadas anteriores, quatro.
with (saida / "registro.csv").open(encoding="utf-8") as f:
    novas = sorted({linha[0] for linha in csv.reader(f) if len(linha) > 4})
ano = {a.stem: a.parent.parent.name for a in coleta.glob("*/html/*.html")}

pacote = saida.parent / f"{saida.name}_novas.zip"
with zipfile.ZipFile(pacote, "w", zipfile.ZIP_DEFLATED) as z:
    for nome in ("registro.csv", "longas.txt"):
        z.write(saida / nome, f"{saida.name}/{nome}")
    for slug in novas:
        for pasta, extensao in (("bruto", "txt"), ("json", "json")):
            arquivo = saida / pasta / ano[slug] / f"{slug}.{extensao}"
            if arquivo.exists():   # o JSON só existe quando a resposta é válida
                z.write(arquivo, f"{saida.name}/{pasta}/{ano[slug]}/{slug}.{extensao}")
print(f"{len(novas)} leis em {pacote}")